In [1]:
import sys
sys.path.append("../")

import pandas as pd
import os
from app.services.skill_taxonomy import extract_skills_from_text, SKILL_TAXONOMY
from app.services.resume_parser import get_resume_text

df = pd.read_csv("../data/Resume.csv")
print(f"Taxonomy has {len(SKILL_TAXONOMY)} known skills")
df.head()

Taxonomy has 254 known skills


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [2]:
def safe_get_text(row=None, file_path=None):
    """
    Tries file-based extraction first (if file_path given and exists),
    otherwise falls back to the CSV's Resume_str column.
    Never raises — returns "" on total failure.
    """
    if file_path and os.path.exists(file_path):
        ext = os.path.splitext(file_path)[1].lower()
        try:
            with open(file_path, "rb") as f:
                file_bytes = f.read()
            text = get_resume_text(file_bytes=file_bytes, file_extension=ext)
            if text:
                return text
        except Exception:
            pass

    if row is not None:
        return get_resume_text(raw_text=row.get("Resume_str", ""))

    return ""

In [3]:
sample_df = df.sample(n=20, random_state=42).copy()

results = []
for _, row in sample_df.iterrows():
    text = safe_get_text(row=row, file_path=None)
    skills = extract_skills_from_text(text)
    results.append({
        "ID": row["ID"],
        "category": row["Category"],
        "num_skills": len(skills),
        "skills": skills,
    })

results_df = pd.DataFrame(results)
results_df

,ID,category,num_skills,skills
0,99244405,TEACHER,3,"[communication, leadership, problem solving]"
1,17562754,DIGITAL-MEDIA,2,"[leadership, product management]"
2,30311725,CONSTRUCTION,5,"[communication, leadership, problem solving, p..."
3,19007667,CHEF,1,[chef]
4,11065180,BANKING,5,"[communication, critical thinking, excel, prob..."
5,39237915,BUSINESS-DEVELOPMENT,3,"[communication, leadership, project management]"
6,17199951,DESIGNER,1,[excel]
7,18236085,BUSINESS-DEVELOPMENT,2,"[communication, leadership]"
8,79663360,TEACHER,2,"[communication, leadership]"
9,62312955,DESIGNER,3,"[express, r, vue]"


In [4]:
def count_skills_safe(row):
    text = safe_get_text(row=row, file_path=None)
    return len(extract_skills_from_text(text))

df["num_skills_found"] = df.apply(count_skills_safe, axis=1)
df.groupby("Category")["num_skills_found"].mean().sort_values()

Category
TEACHER                   2.303922
FITNESS                   2.384615
ACCOUNTANT                2.576271
ARTS                      2.601942
SALES                     2.715517
APPAREL                   2.752577
CHEF                      2.788136
BUSINESS-DEVELOPMENT      2.816667
CONSTRUCTION              2.928571
AVIATION                  2.940171
ADVOCATE                  2.949153
FINANCE                   3.101695
PUBLIC-RELATIONS          3.135135
DESIGNER                  3.224299
HR                        3.236364
HEALTHCARE                3.304348
AGRICULTURE               3.317460
BPO                       3.500000
BANKING                   3.643478
DIGITAL-MEDIA             3.947917
AUTOMOBILE                4.055556
CONSULTANT                4.756522
ENGINEERING               5.169492
INFORMATION-TECHNOLOGY    6.175000
Name: num_skills_found, dtype: float64

In [5]:
worst_category = df.groupby("Category")["num_skills_found"].mean().idxmin()
print(f"Worst category: {worst_category}")

sample_row = df[df["Category"] == worst_category].iloc[0]
text = safe_get_text(row=sample_row, file_path=None)

print(text[:1500])
print("\n--- Detected skills ---")
print(extract_skills_from_text(text))

Worst category: TEACHER
TEACHER         Professional Summary     Master teacher looking for new role and Industry. I'm looking to leverage the valuable skills, knowledge, and experiences as a teacher to advance a new client or organization in an executive or leadership position.       Skills          Instructional Design, Teaching, Progress Monitoring, Course Development  Evernote, Slack, Social Media, Excel, Outlook, Adobe Photoshop  Management: Staff, Projects, Daily Scheduling      Financial Analysis, Security Trading, Portfolio Managament, Liability Structuring  Research, Analysis, and Planning            Work History      Teacher  ,     08/2014   to   Current     Company Name   –   City  ,   State      Teaches classes in accordance with requirements of approved courses of study at expected student progress expectations  Uses information about individual students' academic strengths, needs, and progress in planning  Designs activities to engage students in cognitively challenging w

In [6]:
test_file_path = "../data/raw/sample_resume.pdf"  # change path or leave as-is if none exists

text_from_file = safe_get_text(row=None, file_path=test_file_path)
if text_from_file:
    print("Loaded from FILE — skills found:")
    print(extract_skills_from_text(text_from_file))
else:
    print(f"No file at {test_file_path} — fallback returned empty string safely, no error.")

No file at ../data/raw/sample_resume.pdf — fallback returned empty string safely, no error.


In [7]:
# After eyeballing misses above, add relevant new skills here,
# then copy them into app/services/skill_taxonomy.py permanently
new_skills_to_add = {
    # e.g. "sap", "salesforce", "quickbooks", "autocad"
}
print(f"Proposed additions: {new_skills_to_add}")

Proposed additions: {}
